# Create regression input dataset for GDP / capita model

In [3]:
import xarray as xr
import xesmf as xe
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
import xagg as xa
import os
import re
import glob
import calendar
from matplotlib import pyplot as plt
from cartopy import crs as ccrs
import cmocean

import warnings
from funcs_support import get_params,get_filepaths,utility_save,utility_print
from funcs_aux import _remove_chunk_encoding

from funcs_plot import plot_map,vals_to_color_triangle
dir_list = get_params()
from fsspec.implementations import local
fs = local.LocalFileSystem()

In [4]:
from distributed import Client

# Start dask client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44193 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44193/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44193/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42059,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44193/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43373,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44689/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:33131,


In [5]:
# Load reference dataset
df_dta = pd.read_csv(dir_list['aux']+'GrowthClimateDataset.csv')

years_insamp = np.unique(df_dta.year.values)

df_dta = df_dta.set_index(['countryname','year'])

In [6]:
# Load shapefile for insample regions
gdf = gpd.read_file(dir_list['aux']+'geo_data/country.shp')

rename_keys = {'Belarus':'Byelarus',
 'Brunei Darussalam':'Brunei',
 'Cabo Verde':'Cape Verde',
 'Channel Islands':['Jersey','Guernsey'],
 'Congo, Dem. Rep.':'Zaire',
 'Congo, Rep.':'Congo',
 "Cote d'Ivoire":'Ivory Coast',
 'Egypt, Arab Rep.':'Egypt',
 'Hong Kong SAR, China':'Hong Kong', #probably missing,
 'Iran, Islamic Rep.':'Iran',
 'Isle of Man':'Man, Isle of',
 'Korea, Rep.':'South Korea',
 'Kosovo':'Kosovo', # still part of Serbia in this sample
 'Kyrgyz Republic':'Kyrgyzstan',
 'Lao PDR':'Laos',
 'Macao SAR, China':'Macau',
 'Macedonia, FYR':'Macedonia',
 'Micronesia, Fed. Sts.':'Federated States of Micronesia',
 'Palau':'Pacific Islands (Palau)',
 'Russian Federation':'Russia',
 'Samoa':'Western Samoa',
 'Slovak Republic':'Slovakia',
 'Syrian Arab Republic':'Syria',
 'Tanzania':'Tanzania, United Republic of',
 'Timor-Leste':'Timor-Leste', # uh, still part of indonesia in this one... 
 'Venezuela, RB':'Venezuela',
 'West Bank and Gaza':['West Bank','Gaza Strip'],
 'Yemen, Rep.':'Yemen'}
# Switch order
rename_keys = {(tuple(v) if type(v) == list else v):k for k,v in rename_keys.items()}

# Combine into single the ones that are multiple locs in the shapefile but not the repl dataset
combine_locs = {k:v for k,v in rename_keys.items() if type(k) == tuple}
for locs,joint_name in combine_locs.items():
    gdf_tmp = gdf.iloc[[c in locs for c in gdf.CNTRY_NAME.values],:]
    cols = [col for col in gdf_tmp.columns if col != 'geometry']
    
    gdf_joint = pd.concat([gdf_tmp.iloc[0].loc[[col for col in cols if gdf_tmp.dtypes[col] == object]],
                          gdf_tmp.loc[:,[col for col in cols if gdf_tmp.dtypes[col] in [float,int]]].sum()])
    gdf_joint = gpd.GeoDataFrame([gdf_joint],index=[gdf_tmp.index[0]],
                     geometry = [gdf_tmp.union_all()],
                                 crs = gdf_tmp.crs)
    gdf_joint.loc[:,'CNTRY_NAME'] = joint_name
    
    gdf = pd.concat([gdf.iloc[[c not in locs for c in gdf.CNTRY_NAME.values],:],
               gdf_joint])
gdf = gdf.sort_index()

# Rename the rest of the locations as needed
for idx in gdf.index:
    if gdf.loc[idx,'CNTRY_NAME'] in rename_keys:
        gdf.loc[idx,'CNTRY_NAME'] = rename_keys[gdf.loc[idx,'CNTRY_NAME']]

In [7]:
# Load population density
pop = xr.open_dataset(dir_list['aux']+'pop_data/glp00ag30.asc')
pop = xa.fix_ds(pop)
pop = pop.isel(band=0)

In [8]:
df = get_filepaths()

for obs_mod in ['ERA5-025','MERRA2','JRA-3Q','GMFD','UDel','ERA5-025-1deg','MERRA2-1deg','JRA-3Q-1deg','GMFD-1deg','UDel-1deg']:
    print(f'\nProcessing {obs_mod}')
    if obs_mod+'_temp_popweight' in df_dta.columns:
        print(f'{obs_mod} already processed, skipped!')
        continue
    df_tmp = df.query(f'model == "{obs_mod}" and (varname == "tas" or varname == "pr") and freq == "day"')
    if len(df_tmp) == 0:
        df_tmp = df.query(f'model == "{obs_mod}" and (varname == "tas" or varname == "pr") and freq == "Amon"')
    
    # Load climate data
    dss = {row[1]['varname']:xr.open_dataset(row[1]['path'],chunks='auto') for row in df_tmp.iterrows()}
    dss = {var:ds.isel(time=[y in years_insamp for y in ds.time.dt.year.values])
           for var,ds in dss.items()}
    
    # Subset to global land
    params_subset = {'lat':slice(-58,89),
                     'lon':slice(-180,180)}
    dss = {var:ds.sel(**params_subset) for var,ds in dss.items()}
    
    # Convert precip data from kg/m^2s to mm/day
    if obs_mod not in ['UDel','UDel-1deg']:
        dss['pr'] = dss['pr']*60*60*24 # from kg/m^2s to mm/day
    
    # Convert t from K to T
    dss['tas'] = dss['tas'] - 273.15
    
    # Calculate mean temperature and summed precipitation
    dss = {'tas':dss['tas'].resample(time='1YS').mean(),
           'pr':dss['pr'].resample(time='1YS').sum()}
    
    # Convert time dimension to just year
    for var in dss:
        dss[var]['time'] = dss[var]['time'].dt.year
        dss[var] = dss[var].rename({'time':'year'})

    # Merge
    dss = xr.merge([ds for var,ds in dss.items()])
    
    # Compute means/sums
    with warnings.catch_warnings():
        # Large graph size dask warning
        warnings.filterwarnings('ignore')
        dss = dss.compute()
    
    # (there's no subnational nonlinearity in this model anyways, but for consistency,)
    # Regrid to 0.25 x 0.25 ref grid, if not 1-deg 
    if '1deg' not in obs_mod:
        fine_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.125,
    														 params_subset['lat'].stop-0.124,0.25)),
    								'lon':(['lon'],np.arange(params_subset['lon'].start+0.125,
    														 params_subset['lon'].stop-0.124,0.25))})

    
        rgrd = xe.Regridder(dss,fine_grid,'bilinear')
        dss = rgrd(dss)

    # For some reason lat_bnds still in file
    dss = dss.drop_vars('lat_bnds',errors='ignore')
    
    # Aggregate
    with warnings.catch_warnings():
        # Weights bbox warning - fine, affects uninhabited arctic, and large graph size dask warning
        warnings.filterwarnings('ignore') 
        # Otherwise divide by 0 error in xa.aggregate, should open a github issue
        wm = xa.pixel_overlaps(dss,gdf,weights=pop.band_data.where(pop.band_data>0,0.0000000000001))
        
        with xa.set_options(impl='numba',silent=True):
            agg = xa.aggregate(dss,wm)
    
    # Put aggregated data into correct format for merging
    agg_out = (agg.to_dataframe()[['CNTRY_NAME','tas','pr']].
     rename({'CNTRY_NAME':'countryname'},axis=1).
     reset_index().set_index(['countryname','year']).
     drop('poly_idx',axis=1))
    agg_out = agg_out.rename({'tas':f'{obs_mod}_temp_popweight',
                              'pr':f'{obs_mod}_precip_popweight'},
                             axis=1)
    
    # Calculate squared precip
    agg_out[f'{obs_mod}_precip_popweight_2'] = agg_out[f'{obs_mod}_precip_popweight']**2
    
    
    # Join into existing dataset
    df_dta = df_dta.join(agg_out,how='outer')


Processing ERA5-025
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!

Processing MERRA2
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!

Processing JRA-3Q
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!

Processing GMFD
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!

Processing UDel
UDel already processed, skipped!

Processing ERA5-025-1deg
creating polygons for each pixel...
lat/lon bounds not f

In [9]:
# Drop rows with no observations
df_dta = df_dta.where(~np.isnan(df_dta.Pop)).dropna(how='all')

In [10]:
# Switch climate column name order
df_dta.columns = [
    c if "popweight" not in c else "_".join(c.split("_")[1:] + [c.split("_")[0]])
    for c in df_dta.columns
]

In [11]:
# Identify variable columns using the known pattern
clim_cols = [c for c in df_dta.reset_index().columns 
              if 'popweight' in c] 

# Melt them
df_long = df_dta.reset_index().melt(
    id_vars=[c for c in df_dta.reset_index().columns if c not in clim_cols],
    value_vars=clim_cols,
    var_name='tmp',
    value_name='value'
)

In [12]:
# Split "temp_popweight_UDel" → var, weight, model
df_long[['var','typ','obs_mod']] = df_long['tmp'].str.rsplit('_', n=2, expand=True)
# Get full variable name back
df_long['var_full'] = df_long['var'] + '_' + df_long['typ']

In [13]:
# Pivot back to wide, with clim variables their own thing now
with warnings.catch_warnings():
    # DataFrame is highly fragmented warning from pandas
    warnings.filterwarnings('ignore')
    df_wide = df_long.pivot_table(
        index=['obs_mod'] + [c for c in df_dta.reset_index().columns if c not in clim_cols],
        columns='var_full',
        values='value'
    ).reset_index()

# Get rid of 'var_full' column namer
df_wide.columns.name = None

# Set indices
df_wide = df_wide.set_index(['countryname','year','obs_mod'])

In [14]:
df_out = (df_long.loc[:,['countryname','year','var_full','obs_mod','value']].pivot_table(
    index = ['countryname','year','obs_mod'],
    columns = 'var_full',
    values = 'value').
 join(df_dta.loc[:,[c for c in df_dta.reset_index().columns 
                    if c not in ['countryname','year',*clim_cols]]]))

In [15]:
df_out['nobs_mods'] = df_out.groupby(['countryname','year']).count()['precip_popweight']

In [19]:
df_out_025deg = df_out.iloc[['1deg' not in m for m in df_out.index.get_level_values('obs_mod')],:]
df_out_1deg = df_out.iloc[['1deg' in m for m in df_out.index.get_level_values('obs_mod')],:]

In [20]:
# Export 0.25-deg data
df_out_025deg.to_csv(dir_list['proc']+'reg_inputs/gdppc/reginputs_gdppc_025deg.csv')

# Make the model names identical , without the "1deg" attachment
df_out_1deg = df_out_1deg.reset_index()
df_out_1deg['obs_mod'] = [re.sub(r'\-1deg','',mod) for mod in df_out_1deg.obs_mod.values]
df_out_1deg = df_out_1deg.set_index(['countryname','year','obs_mod'])
# Export 1-deg data
df_out_1deg.to_csv(dir_list['proc']+'reg_inputs/gdppc/reginputs_gdppc_1deg.csv')
